# Assignment 6 — Build Your Own Content-Based Recommender
**Domain:** Food / Recipes  
**Dataset:** https://www.kaggle.com/datasets/shuyangli94/food-com-recipes-and-user-interactions

 

This notebook implements a content-based recommender for recipes and aligns with the assignment requirements:

- use item metadata with at least one text field
- build multiple text vectorization methods (BoW, TF-IDF, BERT/SentenceTransformer)
- optionally combine text with numerical and categorical features
- create an end-to-end recommender using user feedback
- compare against non-personalized and collaborative filtering baselines
- simulate chatbot-driven recommendations

## 1. Environment setup

This notebook is designed for **Google Colab**.

### Data expected
Download the **Food.com Recipes and Interactions** dataset from Kaggle and upload the files to Colab, or mount Google Drive.

Common files used in this notebook:
- `RAW_recipes.csv`
- `RAW_interactions.csv`

If your local filenames differ, update the paths below.

In [ ]:


import ast
import gc
import json
import math
import os
import random
import re
import warnings
from collections import defaultdict

import numpy as np
import pandas as pd

from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import MultiLabelBinarizer, MinMaxScaler
from sklearn.model_selection import train_test_split

warnings.filterwarnings("ignore")
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
random.seed(RANDOM_STATE)

In [ ]:
import nltk
nltk.download("stopwords")
nltk.download("wordnet")
nltk.download("omw-1.4")
nltk.download("punkt")
nltk.download("punkt_tab")

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\grego\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\grego\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\grego\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\grego\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\grego\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

## 2. Load the Food.com dataset

In [82]:
# Update these paths if needed
RECIPES_PATH = "RAW_recipes.csv"
INTERACTIONS_PATH = "RAW_interactions.csv"

recipes = pd.read_csv(RECIPES_PATH)
interactions = pd.read_csv(INTERACTIONS_PATH)

print("recipes shape:", recipes.shape)
print("interactions shape:", interactions.shape)
display(recipes.head(2))
display(interactions.head(2))

recipes shape: (231637, 12)
interactions shape: (1132367, 5)


,name,id,minutes,contributor_id,submitted,tags,nutrition,n_steps,steps,description,ingredients,n_ingredients
0,arriba baked winter squash mexican style,137739,55,47892,2005-09-16,"['60-minutes-or-less', 'time-to-make', 'course...","[51.5, 0.0, 13.0, 0.0, 2.0, 0.0, 4.0]",11,"['make a choice and proceed with recipe', 'dep...",autumn is my favorite time of year to cook! th...,"['winter squash', 'mexican seasoning', 'mixed ...",7
1,a bit different breakfast pizza,31490,30,26278,2002-06-17,"['30-minutes-or-less', 'time-to-make', 'course...","[173.4, 18.0, 0.0, 17.0, 22.0, 35.0, 1.0]",9,"['preheat oven to 425 degrees f', 'press dough...",this recipe calls for the crust to be prebaked...,"['prepared pizza crust', 'sausage patty', 'egg...",6


,user_id,recipe_id,date,rating,review
0,38094,40893,2003-02-17,4,Great with a salad. Cooked on top of stove for...
1,1293707,40893,2011-12-21,5,"So simple, so delicious! Great for chilly fall..."


## 3. Initial inspection

In [83]:
print(recipes.columns.tolist())
print(interactions.columns.tolist())

['name', 'id', 'minutes', 'contributor_id', 'submitted', 'tags', 'nutrition', 'n_steps', 'steps', 'description', 'ingredients', 'n_ingredients']
['user_id', 'recipe_id', 'date', 'rating', 'review']


In [84]:
recipes.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 231637 entries, 0 to 231636
Data columns (total 12 columns):
 #   Column          Non-Null Count   Dtype 
---  ------          --------------   ----- 
 0   name            231636 non-null  object
 1   id              231637 non-null  int64 
 2   minutes         231637 non-null  int64 
 3   contributor_id  231637 non-null  int64 
 4   submitted       231637 non-null  object
 5   tags            231637 non-null  object
 6   nutrition       231637 non-null  object
 7   n_steps         231637 non-null  int64 
 8   steps           231637 non-null  object
 9   description     226658 non-null  object
 10  ingredients     231637 non-null  object
 11  n_ingredients   231637 non-null  int64 
dtypes: int64(5), object(7)
memory usage: 21.2+ MB


In [85]:
interactions.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1132367 entries, 0 to 1132366
Data columns (total 5 columns):
 #   Column     Non-Null Count    Dtype 
---  ------     --------------    ----- 
 0   user_id    1132367 non-null  int64 
 1   recipe_id  1132367 non-null  int64 
 2   date       1132367 non-null  object
 3   rating     1132367 non-null  int64 
 4   review     1132198 non-null  object
dtypes: int64(3), object(2)
memory usage: 43.2+ MB


## 4. Data cleaning and feature engineering


In [86]:
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

stop_words = set(stopwords.words("english"))
lemmatizer = WordNetLemmatizer()

def safe_literal_eval(x):
    if pd.isna(x):
        return []
    if isinstance(x, list):
        return x
    try:
        return ast.literal_eval(x)
    except Exception:
        return []

def clean_text(text):
    text = "" if pd.isna(text) else str(text).lower()
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    tokens = text.split()
    tokens = [lemmatizer.lemmatize(tok) for tok in tokens if tok not in stop_words and len(tok) > 1]
    return " ".join(tokens)

recipes = recipes.copy()
for col in ["tags", "nutrition", "steps", "ingredients"]:
    if col in recipes.columns:
        recipes[col] = recipes[col].apply(safe_literal_eval)

# Basic cleaning
for col in ["name", "description"]:
    if col in recipes.columns:
        recipes[col] = recipes[col].fillna("")

recipes = recipes.drop_duplicates(subset=["id"]).reset_index(drop=True)

# Parse list-like columns into readable strings
recipes["tags_text"] = recipes["tags"].apply(lambda x: " ".join(map(str, x)) if isinstance(x, list) else "")
recipes["ingredients_text"] = recipes["ingredients"].apply(lambda x: " ".join(map(str, x)) if isinstance(x, list) else "")
recipes["steps_text"] = recipes["steps"].apply(lambda x: " ".join(map(str, x)) if isinstance(x, list) else "")

# Nutrition is usually a fixed-length list in Food.com
def nutrition_value(values, idx):
    if isinstance(values, list) and len(values) > idx:
        try:
            return float(values[idx])
        except Exception:
            return np.nan
    return np.nan

nutrition_names = ["calories", "total_fat", "sugar", "sodium", "protein", "sat_fat", "carbs"]
for i, col in enumerate(nutrition_names):
    recipes[col] = recipes["nutrition"].apply(lambda x, idx=i: nutrition_value(x, idx))

recipes["combined_text_raw"] = (
    recipes["name"].fillna("") + " "
    + recipes["description"].fillna("") + " "
    + recipes["ingredients_text"].fillna("") + " "
    + recipes["steps_text"].fillna("") + " "
    + recipes["tags_text"].fillna("")
)

recipes["combined_text"] = recipes["combined_text_raw"].apply(clean_text)

recipes[["id", "name", "combined_text"]].head()

,id,name,combined_text
0,137739,arriba baked winter squash mexican style,arriba baked winter squash mexican style autum...
1,31490,a bit different breakfast pizza,bit different breakfast pizza recipe call crus...
2,112140,all in the kitchen chili,kitchen chili modified version mom chili hit 2...
3,59389,alouette potatoes,alouette potato super easy great tasting make ...
4,44061,amish tomato ketchup for canning,amish tomato ketchup canning dh amish mother r...


In [87]:
# Keep recipes that have meaningful text
recipes = recipes[recipes["combined_text"].str.len() > 20].reset_index(drop=True)

# Optional sampling for faster experimentation in Colab
MAX_RECIPES = 12000
if len(recipes) > MAX_RECIPES:
    recipes = recipes.sample(MAX_RECIPES, random_state=RANDOM_STATE).reset_index(drop=True)

print("Filtered recipes:", recipes.shape)

Filtered recipes: (12000, 24)


## 5. Prepare interaction data for end-to-end recommendations

We keep users with enough interactions and recipes that remain in our filtered item set.

In [88]:
interactions = interactions.copy()
interactions = interactions.rename(columns={"recipe_id": "id"})
interactions = interactions[interactions["id"].isin(recipes["id"])].copy()

# Keep explicit positive feedback; adjust threshold if desired
if "rating" in interactions.columns:
    interactions = interactions[interactions["rating"] >= 4].copy()

# Filter sparse users
user_counts = interactions["user_id"].value_counts()
valid_users = user_counts[user_counts >= 5].index
interactions = interactions[interactions["user_id"].isin(valid_users)].copy()

print("Filtered interactions:", interactions.shape)
print("Unique users:", interactions["user_id"].nunique())
print("Unique recipes in interactions:", interactions["id"].nunique())
display(interactions.head())

Filtered interactions: (25814, 5)
Unique users: 1522
Unique recipes in interactions: 8791


,user_id,id,date,rating,review
16,68960,200236,2006-12-19,4,We really enjoyed this and so did both my youn...
217,364211,435357,2012-10-21,5,This does have a nice fresh taste. I used cra...
218,2000431901,435357,2016-02-26,4,Nice and easy. I used one big grocery store eg...
244,107583,373842,2009-12-05,5,Delicious quiche! I didn't use a frozen crust...
247,461834,373842,2013-09-22,5,Wonderful quiche!! I made 1/3 of the recipe a...


## 6. Build multiple text vectorization methods

The assignment explicitly asks for multiple text vectorization techniques.  
We implement:

1. **Bag of Words**
2. **TF-IDF**
3. **BERT-style sentence embeddings** via `sentence-transformers`

To keep Colab responsive, you can reduce the recipe sample size above.

In [89]:
recipe_index = recipes.reset_index(drop=True).copy()
id_to_idx = {rid: idx for idx, rid in enumerate(recipe_index["id"].tolist())}
idx_to_id = {idx: rid for rid, idx in id_to_idx.items()}
texts = recipe_index["combined_text"].tolist()

In [90]:
# BoW
bow_vectorizer = CountVectorizer(max_features=12000, ngram_range=(1, 2), min_df=2)
X_bow = bow_vectorizer.fit_transform(texts)
print("BoW matrix:", X_bow.shape)

# TF-IDF
tfidf_vectorizer = TfidfVectorizer(max_features=12000, ngram_range=(1, 2), min_df=2)
X_tfidf = tfidf_vectorizer.fit_transform(texts)
print("TF-IDF matrix:", X_tfidf.shape)

BoW matrix: (12000, 12000)
TF-IDF matrix: (12000, 12000)


In [91]:
# BERT / sentence embeddings
# Run encoding in a subprocess so the notebook kernel does not crash on local DLL issues.
bert_available = False
bert_source = "fallback"
bert_model_name = "sentence-transformers/all-MiniLM-L6-v2"


def _row_normalize_dense(x):
    denom = np.linalg.norm(x, axis=1, keepdims=True) + 1e-12
    return x / denom


def encode_bert_in_subprocess(text_items, model_name=bert_model_name, batch_size=64):
    import subprocess
    import sys
    import tempfile

    with tempfile.NamedTemporaryFile(mode="w", suffix=".json", delete=False, encoding="utf-8") as f_in:
        json.dump(text_items, f_in)
        in_path = f_in.name

    with tempfile.NamedTemporaryFile(suffix=".npy", delete=False) as f_out:
        out_path = f_out.name

    script = r'''
import json
import numpy as np
import sys
from sentence_transformers import SentenceTransformer

in_path, out_path, model_name, batch_size = sys.argv[1], sys.argv[2], sys.argv[3], int(sys.argv[4])
with open(in_path, "r", encoding="utf-8") as f:
    texts = json.load(f)

model = SentenceTransformer(model_name)
emb = model.encode(
    texts,
    show_progress_bar=True,
    batch_size=batch_size,
    convert_to_numpy=True,
    normalize_embeddings=True,
)
np.save(out_path, emb)
print(emb.shape)
'''

    try:
        result = subprocess.run(
            [sys.executable, "-c", script, in_path, out_path, model_name, str(batch_size)],
            capture_output=True,
            text=True,
            check=False,
        )
        if result.returncode != 0:
            raise RuntimeError((result.stderr or result.stdout).strip())
        emb = np.load(out_path)
        return emb
    finally:
        for p in [in_path, out_path]:
            try:
                os.remove(p)
            except OSError:
                pass


try:
    try:
        X_bert = encode_bert_in_subprocess(texts, model_name=bert_model_name, batch_size=64)
    except ModuleNotFoundError:
        import subprocess
        import sys

        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "sentence-transformers"])
        X_bert = encode_bert_in_subprocess(texts, model_name=bert_model_name, batch_size=64)

    bert_available = True
    bert_source = "sentence-transformers"
    print("BERT matrix:", X_bert.shape)
except Exception as bert_err:
    print(f"BERT unavailable ({type(bert_err).__name__}: {bert_err})")
    print("Falling back to normalized TF-IDF vectors for BERT-dependent sections.")
    X_bert = X_tfidf.astype(np.float32).toarray()
    X_bert = _row_normalize_dense(X_bert)
    bert_source = "tfidf_fallback"
    print("Fallback matrix:", X_bert.shape)

print(f"BERT source: {bert_source}")

Exception in thread Thread-6 (_readerthread):
Traceback (most recent call last):
  File "c:\Users\grego\AppData\Local\Programs\Python\Python312\Lib\threading.py", line 1073, in _bootstrap_inner
    self.run()
  File "C:\Users\grego\AppData\Roaming\Python\Python312\site-packages\ipykernel\ipkernel.py", line 761, in run_closure
    _threading_Thread_run(self)
  File "c:\Users\grego\AppData\Local\Programs\Python\Python312\Lib\threading.py", line 1010, in run
    self._target(*self._args, **self._kwargs)
  File "c:\Users\grego\AppData\Local\Programs\Python\Python312\Lib\subprocess.py", line 1597, in _readerthread
    buffer.append(fh.read())
                  ^^^^^^^^^
  File "c:\Users\grego\AppData\Local\Programs\Python\Python312\Lib\encodings\cp1252.py", line 23, in decode
    return codecs.charmap_decode(input,self.errors,decoding_table)[0]
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
UnicodeDecodeError: 'charmap' codec can't decode byte 0x8f in position 799: chara

BERT matrix: (12000, 384)
BERT source: sentence-transformers


## 7. Add numerical and categorical features

This section creates extra features that can be combined with text similarities.

In [92]:
numeric_cols = [c for c in ["minutes"] + nutrition_names if c in recipe_index.columns]
num_df = recipe_index[numeric_cols].copy()

for c in numeric_cols:
    num_df[c] = pd.to_numeric(num_df[c], errors="coerce")

num_df = num_df.fillna(num_df.median(numeric_only=True))
scaler = MinMaxScaler()
X_num = scaler.fit_transform(num_df)

# Multi-label tags
mlb = MultiLabelBinarizer()
tag_lists = recipe_index["tags"].apply(lambda x: x if isinstance(x, list) else [])
X_tags = mlb.fit_transform(tag_lists)

print("Numeric matrix:", X_num.shape)
print("Tag matrix:", X_tags.shape)

Numeric matrix: (12000, 8)
Tag matrix: (12000, 483)


## 8. Similarity functions and recommenders

Note: This directly connects to Savory, where we turn food content into structured recommendations.

In [93]:
def get_similarity_matrix(X, metric="cosine"):
    if metric != "cosine":
        raise ValueError("Only cosine is implemented here.")
    return cosine_similarity(X)

sim_bow = cosine_similarity(X_bow)
sim_tfidf = cosine_similarity(X_tfidf)
# X_bert is normalized (or fallback-normalized), so dot product equals cosine similarity.
sim_bert = X_bert @ X_bert.T
sim_num = cosine_similarity(X_num)
sim_tags = cosine_similarity(X_tags) if X_tags.shape[1] > 0 else np.zeros((len(recipe_index), len(recipe_index)))

def hybrid_similarity(alpha_text=0.7, alpha_num=0.2, alpha_tags=0.1, base="tfidf"):
    text_sim = {"bow": sim_bow, "tfidf": sim_tfidf, "bert": sim_bert}[base]
    return alpha_text * text_sim + alpha_num * sim_num + alpha_tags * sim_tags

hybrid_base = "bert" if bert_available else "tfidf"
sim_hybrid = hybrid_similarity(alpha_text=0.7, alpha_num=0.2, alpha_tags=0.1, base=hybrid_base)
print(f"Hybrid text base: {hybrid_base}")

def recommend_similar(recipe_id, sim_matrix, top_k=10):
    idx = id_to_idx.get(recipe_id)
    if idx is None:
        raise ValueError("Recipe ID not found.")
    scores = list(enumerate(sim_matrix[idx]))
    scores = sorted(scores, key=lambda x: x[1], reverse=True)
    scores = [x for x in scores if x[0] != idx][:top_k]
    out = recipe_index.iloc[[i for i, _ in scores]][["id", "name", "minutes", "tags_text"]].copy()
    out["score"] = [s for _, s in scores]
    return out

Hybrid text base: bert


In [94]:
# Example content-based recommendation
example_recipe_id = recipe_index.iloc[0]["id"]
print("Query recipe:", recipe_index.iloc[0]["name"])
recommend_similar(example_recipe_id, sim_tfidf, top_k=5)

Query recipe: crab filled crescent snacks


,id,name,minutes,tags_text,score
2125,180225,pizza dogs,30,30-minutes-or-less time-to-make course main-in...,0.397148
7324,185648,bacon crescents,40,60-minutes-or-less time-to-make course prepara...,0.355553
11581,295380,simple cinnamon crescent rolls,30,30-minutes-or-less time-to-make course prepara...,0.306764
2495,126994,candy bar croissant,25,30-minutes-or-less time-to-make course main-in...,0.298427
1732,24188,mini crescent stromboli rolls,28,ham 30-minutes-or-less time-to-make course mai...,0.295444


## 9. End-to-end recommender based on user preferences

We build user profiles from the last or last-k positively rated items and recommend the most similar unseen recipes.

This closes the loop beyond item-item similarity, as requested in the assignment.

Note: This directly connects to Savory, where we turn food content into structured recommendations.

In [95]:
# Sort by date if available; otherwise preserve current order
date_col = None
for candidate in ["date", "submitted", "review"]:
    if candidate in interactions.columns:
        date_col = candidate
        break

if date_col is not None:
    interactions = interactions.sort_values(["user_id", date_col])

user_histories = interactions.groupby("user_id")["id"].apply(list).to_dict()
eligible_users = [u for u, hist in user_histories.items() if len(hist) >= 5]
len(eligible_users)

1522

In [96]:
def build_user_profile(history_ids, embedding_matrix, k_last=3):
    history_ids = [rid for rid in history_ids if rid in id_to_idx]
    if len(history_ids) == 0:
        return None
    chosen = history_ids[-k_last:]
    idxs = [id_to_idx[rid] for rid in chosen]
    profile = embedding_matrix[idxs].mean(axis=0)
    if profile.ndim == 1:
        return profile
    return np.asarray(profile).mean(axis=0)

def recommend_for_user(user_id, embedding_matrix, top_k=10, k_last=3):
    history = user_histories[user_id]
    seen = set(history)
    profile = build_user_profile(history, embedding_matrix, k_last=k_last)
    if profile is None:
        return pd.DataFrame()
    if len(profile.shape) == 1:
        scores = embedding_matrix @ profile if embedding_matrix.ndim == 2 else cosine_similarity(embedding_matrix, profile.reshape(1, -1)).ravel()
    else:
        scores = cosine_similarity(embedding_matrix, profile).ravel()
    ranked = np.argsort(scores)[::-1]
    ranked = [idx for idx in ranked if idx_to_id[idx] not in seen][:top_k]
    out = recipe_index.iloc[ranked][["id", "name", "minutes", "tags_text"]].copy()
    out["score"] = scores[ranked]
    return out

In [97]:
# Example personalized recommendation
sample_user = eligible_users[0]
print("Sample user:", sample_user)
print("Recent liked recipes:")
display(recipe_index[recipe_index["id"].isin(user_histories[sample_user][-3:])][["id", "name"]])

print("Recommended recipes (BERT profile):")
display(recommend_for_user(sample_user, X_bert, top_k=5, k_last=3))

Sample user: 1533
Recent liked recipes:


,id,name
166,59135,husband s delight
2576,96621,kashmir lamb with spinach
8748,106852,savory onion marmalade


Recommended recipes (BERT profile):


,id,name,minutes,tags_text,score
842,11523,turkey stuffed yellow red bell peppers,65,weeknight time-to-make course main-ingredient ...,0.737201
7496,29503,spinach ravioli with sun dried tomatoes and pi...,25,30-minutes-or-less time-to-make course main-in...,0.733089
9358,83501,creamed onions,55,60-minutes-or-less time-to-make course main-in...,0.729951
3652,482649,sweet tomato chutney,35,60-minutes-or-less time-to-make course main-in...,0.724752
4020,304677,spinach salad with roasted red onions pecans ...,30,30-minutes-or-less time-to-make course cuisine...,0.723100


## 10. Evaluation setup

We test: with a simple leave-one-out strategy:
- for each eligible user, hold out the last liked recipe
- build a user profile from previous liked recipes
- generate top-k recommendations
- measure whether the held-out item appears in the recommendations

Metrics:
- Hit Rate@K
- Recall@K
- MRR@K

This evaluation is straightforward and suitable for recommender assignments.

Note: This directly connects to Savory, where we turn food content into structured recommendations.

In [98]:
def evaluate_user_based_model(embedding_matrix, user_ids, k_last=3, top_k=10):
    hits = []
    reciprocal_ranks = []
    recalls = []

    for user_id in user_ids:
        history = [rid for rid in user_histories[user_id] if rid in id_to_idx]
        if len(history) < k_last + 1:
            continue

        train_hist = history[:-1]
        target = history[-1]
        seen = set(train_hist)

        profile = build_user_profile(train_hist, embedding_matrix, k_last=k_last)
        if profile is None:
            continue

        scores = embedding_matrix @ profile if embedding_matrix.ndim == 2 else cosine_similarity(embedding_matrix, profile.reshape(1, -1)).ravel()
        ranked = np.argsort(scores)[::-1]
        ranked_ids = [idx_to_id[idx] for idx in ranked if idx_to_id[idx] not in seen][:top_k]

        hit = int(target in ranked_ids)
        hits.append(hit)
        recalls.append(hit)  # one relevant held-out item

        if hit:
            rr = 1 / (ranked_ids.index(target) + 1)
        else:
            rr = 0
        reciprocal_ranks.append(rr)

    return {
        f"HitRate@{top_k}": np.mean(hits) if hits else 0,
        f"Recall@{top_k}": np.mean(recalls) if recalls else 0,
        f"MRR@{top_k}": np.mean(reciprocal_ranks) if reciprocal_ranks else 0,
        "n_users_evaluated": len(hits)
    }

In [99]:
eval_users = eligible_users[:1000]  # cap for speed
bert_row_name = "BERT (MiniLM)" if bert_available and bert_source == "sentence-transformers" else "BERT (fallback)"

results = []
for name, emb in [
    ("BoW", X_bow),
    ("TF-IDF", X_tfidf),
    (bert_row_name, X_bert),
]:
    metrics = evaluate_user_based_model(emb, eval_users, k_last=3, top_k=10)
    metrics["model"] = name
    results.append(metrics)

results_df = pd.DataFrame(results)[["model", "HitRate@10", "Recall@10", "MRR@10", "n_users_evaluated"]]
results_df.sort_values("HitRate@10", ascending=False)

,model,HitRate@10,Recall@10,MRR@10,n_users_evaluated
1,TF-IDF,0.005,0.005,0.000729,1000
2,BERT (MiniLM),0.003,0.003,0.000725,1000
0,BoW,0.001,0.001,0.000111,1000


## 11. Hybrid recommender evaluation

We create a hybrid item embedding by concatenating standardized dense representations.

For sparse text matrices like BoW/TF-IDF, we first project them into dense arrays on a manageable sample.

Note: This directly connects to Savory, where we turn food content into structured recommendations.

In [100]:
from scipy import sparse


def dense_concat(text_matrix, numeric_matrix, tag_matrix, text_weight=0.7, num_weight=0.2, tag_weight=0.1):
    if sparse.issparse(text_matrix):
        text_dense = text_matrix.toarray()
    else:
        text_dense = np.asarray(text_matrix)

    num_dense = np.asarray(numeric_matrix)
    tag_dense = np.asarray(tag_matrix)

    # normalize each block row-wise
    def row_norm(x):
        denom = np.linalg.norm(x, axis=1, keepdims=True) + 1e-12
        return x / denom

    text_dense = row_norm(text_dense) * text_weight
    num_dense = row_norm(num_dense) * num_weight
    tag_dense = row_norm(tag_dense) * tag_weight if tag_dense.shape[1] > 0 else tag_dense

    if tag_dense.shape[1] > 0:
        return np.hstack([text_dense, num_dense, tag_dense])
    return np.hstack([text_dense, num_dense])


# Systematic weight search (text / numeric / tags must sum to 1.0)
weight_grid = [
    (0.90, 0.05, 0.05),
    (0.80, 0.15, 0.05),
    (0.80, 0.10, 0.10),
    (0.75, 0.15, 0.10),
    (0.70, 0.20, 0.10),
    (0.70, 0.15, 0.15),
    (0.65, 0.20, 0.15),
    (0.60, 0.25, 0.15),
]

weight_eval_users = eval_users[:400]  # smaller set for faster grid search
weight_search_rows = []

for w_text, w_num, w_tag in weight_grid:
    X_h = dense_concat(X_bert, X_num, X_tags, text_weight=w_text, num_weight=w_num, tag_weight=w_tag)
    m = evaluate_user_based_model(X_h, weight_eval_users, k_last=3, top_k=10)
    m.update({"w_text": w_text, "w_num": w_num, "w_tag": w_tag})
    weight_search_rows.append(m)

weight_search_df = pd.DataFrame(weight_search_rows).sort_values(
    ["HitRate@10", "MRR@10"], ascending=False
).reset_index(drop=True)

best_weights = weight_search_df.iloc[0]
best_text_w = float(best_weights["w_text"])
best_num_w = float(best_weights["w_num"])
best_tag_w = float(best_weights["w_tag"])

print("Best hybrid weights from grid search:", (best_text_w, best_num_w, best_tag_w))
display(weight_search_df)

# Final hybrid with best weights on full evaluation user set
X_hybrid_embed = dense_concat(
    X_bert,
    X_num,
    X_tags,
    text_weight=best_text_w,
    num_weight=best_num_w,
    tag_weight=best_tag_w,
)
hybrid_metrics = evaluate_user_based_model(X_hybrid_embed, eval_users, k_last=3, top_k=10)
hybrid_metrics

Best hybrid weights from grid search: (0.9, 0.05, 0.05)


,HitRate@10,Recall@10,MRR@10,n_users_evaluated,w_text,w_num,w_tag
0,0.0025,0.0025,0.000278,400,0.90,0.05,0.05
1,0.0000,0.0000,0.000000,400,0.80,0.15,0.05
2,0.0000,0.0000,0.000000,400,0.80,0.10,0.10
3,0.0000,0.0000,0.000000,400,0.75,0.15,0.10
4,0.0000,0.0000,0.000000,400,0.70,0.20,0.10
5,0.0000,0.0000,0.000000,400,0.70,0.15,0.15
6,0.0000,0.0000,0.000000,400,0.65,0.20,0.15
7,0.0000,0.0000,0.000000,400,0.60,0.25,0.15


{'HitRate@10': 0.003,
 'Recall@10': 0.003,
 'MRR@10': 0.0007111111111111111,
 'n_users_evaluated': 1000}

## 12. Baselines

The assignment requests comparison against simpler recommenders:
- **Most popular**
- **Random**
- **Collaborative filtering**

These are intentionally minimal and serve as baselines.

Note: This directly connects to Savory, where we turn food content into structured recommendations.

In [101]:
# Unified baseline evaluation (same metrics/protocol for every baseline)
popular_ids = interactions["id"].value_counts().index.tolist()
all_recipe_ids = recipe_index["id"].tolist()


def evaluate_ranker(user_ids, recommend_fn, top_k=10):
    hits, recalls, reciprocal_ranks = [], [], []

    for user_id in user_ids:
        history = [rid for rid in user_histories[user_id] if rid in id_to_idx]
        if len(history) < 2:
            continue

        train_hist = history[:-1]
        target = history[-1]
        recs = recommend_fn(user_id, train_hist, top_k=top_k)

        hit = int(target in recs)
        hits.append(hit)
        recalls.append(hit)  # single held-out relevant item
        reciprocal_ranks.append(1 / (recs.index(target) + 1) if hit else 0)

    return {
        f"HitRate@{top_k}": np.mean(hits) if hits else 0,
        f"Recall@{top_k}": np.mean(recalls) if recalls else 0,
        f"MRR@{top_k}": np.mean(reciprocal_ranks) if reciprocal_ranks else 0,
        "n_users_evaluated": len(hits),
    }


def recommend_popular_eval(user_id, train_hist, top_k=10):
    seen = set(train_hist)
    return [rid for rid in popular_ids if rid not in seen][:top_k]


def recommend_random_eval(user_id, train_hist, top_k=10):
    seen = set(train_hist)
    candidates = [rid for rid in all_recipe_ids if rid not in seen]
    if len(candidates) < top_k:
        return candidates
    rng = random.Random(RANDOM_STATE + int(user_id) % 100000)
    return rng.sample(candidates, top_k)


popular_metrics = evaluate_ranker(eval_users, recommend_popular_eval, top_k=10)
random_metrics = evaluate_ranker(eval_users, recommend_random_eval, top_k=10)

popular_metrics, random_metrics

({'HitRate@10': 0.062,
  'Recall@10': 0.062,
  'MRR@10': 0.02303928571428571,
  'n_users_evaluated': 1000},
 {'HitRate@10': 0.001,
  'Recall@10': 0.001,
  'MRR@10': 0.0001111111111111111,
  'n_users_evaluated': 1000})

In [102]:
# Collaborative filtering baseline with Surprise (evaluated via the same helper)
try:
    try:
        from surprise import Dataset, Reader, SVD
        from surprise.model_selection import train_test_split as surprise_train_test_split
    except ModuleNotFoundError:
        import subprocess
        import sys

        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "scikit-surprise"])
        from surprise import Dataset, Reader, SVD
        from surprise.model_selection import train_test_split as surprise_train_test_split

    cf_df = interactions[["user_id", "id", "rating"]].dropna().copy()
    reader = Reader(rating_scale=(1, 5))
    data = Dataset.load_from_df(cf_df, reader)

    trainset, _ = surprise_train_test_split(data, test_size=0.2, random_state=RANDOM_STATE)
    svd = SVD(random_state=RANDOM_STATE)
    svd.fit(trainset)

    def recommend_cf_eval(user_id, train_hist, top_k=10):
        seen = set(train_hist)
        preds = [(rid, svd.predict(user_id, rid).est) for rid in all_recipe_ids if rid not in seen]
        preds = sorted(preds, key=lambda x: x[1], reverse=True)[:top_k]
        return [rid for rid, _ in preds]

    cf_metrics = evaluate_ranker(eval_users, recommend_cf_eval, top_k=10)
except Exception as cf_err:
    print(f"Collaborative filtering baseline skipped ({type(cf_err).__name__}: {cf_err})")
    cf_metrics = {
        "HitRate@10": np.nan,
        "Recall@10": np.nan,
        "MRR@10": np.nan,
        "n_users_evaluated": 0,
    }

cf_metrics

{'HitRate@10': 0.001,
 'Recall@10': 0.001,
 'MRR@10': 0.0001111111111111111,
 'n_users_evaluated': 1000}

In [103]:
comparison = pd.DataFrame([
    {"model": "Random", **random_metrics},
    {"model": "Popular", **popular_metrics},
    {"model": "BoW", **results_df.set_index("model").loc["BoW"].to_dict()},
    {"model": "TF-IDF", **results_df.set_index("model").loc["TF-IDF"].to_dict()},
    {"model": bert_row_name, **results_df.set_index("model").loc[bert_row_name].to_dict()},
    {"model": "Hybrid (best weights)", **hybrid_metrics},
    {"model": "Collaborative Filtering (SVD)", **cf_metrics},
])

comparison = comparison[["model", "HitRate@10", "Recall@10", "MRR@10", "n_users_evaluated"]]
comparison.sort_values("HitRate@10", ascending=False)

,model,HitRate@10,Recall@10,MRR@10,n_users_evaluated
1,Popular,0.062,0.062,0.023039,1000.0
3,TF-IDF,0.005,0.005,0.000729,1000.0
4,BERT (MiniLM),0.003,0.003,0.000725,1000.0
5,Hybrid (best weights),0.003,0.003,0.000711,1000.0
0,Random,0.001,0.001,0.000111,1000.0
2,BoW,0.001,0.001,0.000111,1000.0
6,Collaborative Filtering (SVD),0.001,0.001,0.000111,1000.0


## 13. Diversity analysis

The assignment mentions analyzing the impact of weighting strategies, especially on diversity.
We use a simple diversity proxy:
- for each user, compute the average pairwise dissimilarity among the recommended items

Higher values indicate more diverse recommendation lists.

In [104]:
def intra_list_diversity(rec_ids, sim_matrix):
    idxs = [id_to_idx[rid] for rid in rec_ids if rid in id_to_idx]
    if len(idxs) < 2:
        return np.nan
    sub = sim_matrix[np.ix_(idxs, idxs)]
    n = len(idxs)
    pair_sims = []
    for i in range(n):
        for j in range(i + 1, n):
            pair_sims.append(sub[i, j])
    if len(pair_sims) == 0:
        return np.nan
    return 1 - np.mean(pair_sims)


def evaluate_diversity_for_embedding(embedding_matrix, user_ids, top_k=10, k_last=3):
    diversities = []

    # Compute the full similarity matrix once per model for speed and stability.
    if hasattr(embedding_matrix, "toarray"):
        sim = cosine_similarity(embedding_matrix)
    else:
        sim = embedding_matrix @ embedding_matrix.T if embedding_matrix.ndim == 2 else cosine_similarity(embedding_matrix)

    for user_id in user_ids[:300]:
        rec_df = recommend_for_user(user_id, embedding_matrix, top_k=top_k, k_last=k_last)
        rec_ids = rec_df["id"].tolist()
        d = intra_list_diversity(rec_ids, sim)
        if not np.isnan(d):
            diversities.append(d)

    return np.mean(diversities) if diversities else np.nan


diversity_scores = {
    "TF-IDF": evaluate_diversity_for_embedding(X_tfidf, eval_users[:300]),
    "BERT": evaluate_diversity_for_embedding(X_bert, eval_users[:300]),
    "Hybrid": evaluate_diversity_for_embedding(X_hybrid_embed, eval_users[:300]),
}
diversity_scores

{'TF-IDF': 0.7470145899775159,
 'BERT': 0.24046633660793304,
 'Hybrid': 0.3814447966230305}

## 14. Chatbot integration (manual + automated)

This section includes both:
- a manual conversation-style query
- an automated chatbot pipeline that turns user text into a recommendation query (LLM-assisted when available, rule-based fallback otherwise).

This directly connects to Savory, where free-text preferences are converted into recipe recommendations.

In [105]:
def encode_query_bert_subprocess(query, model_name=bert_model_name):
    import subprocess
    import sys

    script = r'''
import numpy as np
import sys
from sentence_transformers import SentenceTransformer

query = sys.argv[1]
model_name = sys.argv[2]
model = SentenceTransformer(model_name)
emb = model.encode([query], convert_to_numpy=True, normalize_embeddings=True)
print(','.join(map(str, emb[0].tolist())))
'''
    result = subprocess.run(
        [sys.executable, "-c", script, query, model_name],
        capture_output=True,
        text=True,
        check=False,
    )
    if result.returncode != 0:
        raise RuntimeError((result.stderr or result.stdout).strip())
    values = [float(x) for x in result.stdout.strip().split(",") if x.strip()]
    return np.array(values, dtype=np.float32).reshape(1, -1)


def recommend_from_query(query, vectorizer_or_model, item_matrix, top_k=10, model_type="tfidf"):
    query_clean = clean_text(query)

    if model_type in ["bow", "tfidf"]:
        q_vec = vectorizer_or_model.transform([query_clean])
        sims = cosine_similarity(q_vec, item_matrix).ravel()
    elif model_type == "bert":
        q_vec = encode_query_bert_subprocess(query_clean)
        sims = (q_vec @ item_matrix.T).ravel()
    else:
        raise ValueError("Unsupported model_type")

    top_idx = np.argsort(sims)[::-1][:top_k]
    out = recipe_index.iloc[top_idx][["id", "name", "minutes", "tags_text"]].copy()
    out["score"] = sims[top_idx]
    return out


def summarize_preferences_with_llm(user_message):
    """Optional LLM summarizer. Returns None when API/model is unavailable."""
    try:
        import os
        from openai import OpenAI

        api_key = os.getenv("OPENAI_API_KEY")
        if not api_key:
            return None

        client = OpenAI(api_key=api_key)
        prompt = (
            "Convert the user message into a short recipe-search query with constraints "
            "(diet, meal type, ingredients, time, dislikes). Keep it to one sentence.\n\n"
            f"User message: {user_message}"
        )
        resp = client.responses.create(model="gpt-4o-mini", input=prompt)
        text = (resp.output_text or "").strip()
        return text or None
    except Exception:
        return None


def summarize_preferences_rule_based(user_message):
    msg = user_message.lower()
    tags = []

    if "high protein" in msg or "protein" in msg:
        tags.append("high protein")
    if "vegan" in msg:
        tags.append("vegan")
    if "vegetarian" in msg:
        tags.append("vegetarian")
    if "dairy" in msg or "lactose" in msg:
        tags.append("dairy free")
    if "quick" in msg or "fast" in msg or "30" in msg:
        tags.append("quick")
    if "chicken" in msg:
        tags.append("chicken")
    if "garlic" in msg:
        tags.append("garlic")
    if "salad" in msg:
        tags.append("salad")

    if not tags:
        return user_message
    return " ".join(tags)


def automated_chatbot_recommend(user_message, top_k=10):
    llm_query = summarize_preferences_with_llm(user_message)
    final_query = llm_query if llm_query else summarize_preferences_rule_based(user_message)

    chatbot_model_type = "bert" if bert_available and bert_source == "sentence-transformers" else "tfidf"
    chatbot_model = None if chatbot_model_type == "bert" else tfidf_vectorizer
    chatbot_item_matrix = X_bert if chatbot_model_type == "bert" else X_tfidf

    print("User message:", user_message)
    print("Parsed query:", final_query)
    print(f"Retriever: {chatbot_model_type.upper()}")

    recs = recommend_from_query(
        query=final_query,
        vectorizer_or_model=chatbot_model,
        item_matrix=chatbot_item_matrix,
        top_k=top_k,
        model_type=chatbot_model_type,
    )
    return recs


# Manual conversation-style query
manual_conversation = '''
User: I want a healthy high-protein dinner.
User: I do not want dairy.
User: It should not take too long to cook.
User: Chicken is fine, and I like garlic, herbs, and vegetables.
'''

print("Manual query demo:")
manual_recs = automated_chatbot_recommend(manual_conversation, top_k=10)
display(manual_recs)

# Automated chatbot demo (single natural user message)
auto_user_message = "Need a quick high-protein dinner with chicken and vegetables, no dairy."
print("\nAutomated chatbot demo:")
auto_recs = automated_chatbot_recommend(auto_user_message, top_k=10)
display(auto_recs)

Manual query demo:
User message: 
User: I want a healthy high-protein dinner.
User: I do not want dairy.
User: It should not take too long to cook.
User: Chicken is fine, and I like garlic, herbs, and vegetables.

Parsed query: high protein dairy free chicken garlic
Retriever: BERT


,id,name,minutes,tags_text,score
2990,224197,lemon garlic chicken over angel hair pasta,25,30-minutes-or-less time-to-make course main-in...,0.638786
8795,312820,garlic chicken with orzo,55,60-minutes-or-less time-to-make course main-in...,0.609706
8251,171976,garlic chicken sauce,15,15-minutes-or-less time-to-make course prepara...,0.607861
7341,147475,italian chicken au gratin,40,60-minutes-or-less time-to-make course main-in...,0.605988
9323,83828,chicken in garlic white wine cream sauce,50,60-minutes-or-less time-to-make main-ingredien...,0.592725
179,87789,garlic stuffed chicken breasts,60,60-minutes-or-less time-to-make main-ingredien...,0.589761
10533,406170,outback s roasted garlic herb aioli,10,15-minutes-or-less time-to-make course prepara...,0.581686
5241,202025,garlic lemon chicken breasts,30,30-minutes-or-less time-to-make course main-in...,0.581451
2043,52547,herbed chicken pasta,35,60-minutes-or-less time-to-make main-ingredien...,0.580337
4208,352105,creamy pasta with mushrooms spinach and peas,20,30-minutes-or-less time-to-make course main-in...,0.580030



Automated chatbot demo:
User message: Need a quick high-protein dinner with chicken and vegetables, no dairy.
Parsed query: high protein dairy free quick chicken
Retriever: BERT


,id,name,minutes,tags_text,score
9198,136173,quick chicken gravy,10,15-minutes-or-less time-to-make preparation 5-...,0.582519
7969,159631,peanut chicken,11,15-minutes-or-less time-to-make course main-in...,0.568240
6855,114088,simon and garfunkel chicken,75,time-to-make course main-ingredient preparatio...,0.550819
6094,220292,quick almond chicken,10,15-minutes-or-less time-to-make course main-in...,0.549851
8691,191435,ultra low fat chicken fried chicken with cream...,45,60-minutes-or-less time-to-make main-ingredien...,0.544514
2161,399787,chicken asado,70,lactose time-to-make course main-ingredient pr...,0.544127
5250,136644,easy chicken almondine,50,60-minutes-or-less time-to-make course main-in...,0.543780
1834,90335,super easy chicken and noodles,10,15-minutes-or-less time-to-make course main-in...,0.541309
5508,372049,lemon butter glazed chicken,17,30-minutes-or-less time-to-make course main-in...,0.539081
6069,171061,chicken cardinale,30,30-minutes-or-less time-to-make course main-in...,0.537522


## 15. Qualitative examples

Quick takeaway: you should inspect recommendation lists and comment on:
- whether the recipes are semantically related
- whether nutrition and tags improve relevance
- when BoW, TF-IDF, and BERT behave differently
- whether the chatbot recommendations make practical sense

In [106]:
# Compare recommendation lists from different methods for the same recipe
query_idx = 15
query_recipe = recipe_index.iloc[query_idx]
query_id = query_recipe["id"]

print("Query recipe:")
display(recipe_index.iloc[[query_idx]][["id", "name", "minutes", "tags_text"]])

print("BoW recommendations:")
display(recommend_similar(query_id, sim_bow, top_k=5))

print("TF-IDF recommendations:")
display(recommend_similar(query_id, sim_tfidf, top_k=5))

print("BERT recommendations:")
display(recommend_similar(query_id, sim_bert, top_k=5))

print("Hybrid recommendations:")
display(recommend_similar(query_id, sim_hybrid, top_k=5))

Query recipe:


,id,name,minutes,tags_text
15,377937,haricots verts salad,20,30-minutes-or-less time-to-make course main-in...


BoW recommendations:


,id,name,minutes,tags_text,score
2626,296865,italian sauteed beans,20,30-minutes-or-less time-to-make course main-in...,0.473066
2980,484877,tortellini salad with asparagus and fresh basi...,45,60-minutes-or-less time-to-make course main-in...,0.467574
2087,391189,sage scented ziti with broccoli pine nuts an...,45,60-minutes-or-less time-to-make course prepara...,0.451263
1757,86927,asparagus salad with balsamic vinegar,10,15-minutes-or-less time-to-make course main-in...,0.444000
10739,373342,haricots verts with toasted walnuts and chevre,21,30-minutes-or-less time-to-make course main-in...,0.434643


TF-IDF recommendations:


,id,name,minutes,tags_text,score
5229,113634,snap beans and potatoes,59,60-minutes-or-less time-to-make course main-in...,0.273417
11649,50910,spinach basil salad with pine nut dressing,16,30-minutes-or-less time-to-make course main-in...,0.264064
5450,114023,bruschetta with pesto mozzarella and sun drie...,20,30-minutes-or-less time-to-make course main-in...,0.258572
7460,209591,haricots verts with browned garlic,26,30-minutes-or-less time-to-make course main-in...,0.253527
6849,75454,green beans with ricotta salata,15,15-minutes-or-less time-to-make course main-in...,0.249556


BERT recommendations:


,id,name,minutes,tags_text,score
7460,209591,haricots verts with browned garlic,26,30-minutes-or-less time-to-make course main-in...,0.871110
4643,27603,potato salad with olive tomatoes and capers,45,60-minutes-or-less time-to-make course main-in...,0.816616
7945,268031,veggies in garlic butter,25,30-minutes-or-less time-to-make course main-in...,0.804057
9740,117404,potato and green bean salad with balsamic vina...,60,60-minutes-or-less time-to-make course main-in...,0.792898
10728,76939,great northern bean olive soup,35,60-minutes-or-less time-to-make course main-in...,0.791109


Hybrid recommendations:


,id,name,minutes,tags_text,score
4643,27603,potato salad with olive tomatoes and capers,45,60-minutes-or-less time-to-make course main-in...,0.829979
10489,105739,green beans and tomatoes in a pesto vinaigrette,40,60-minutes-or-less time-to-make course main-in...,0.808942
4295,57290,tomato and basil couscous salad,10,15-minutes-or-less time-to-make course main-in...,0.808736
2640,375073,black bean and corn salad spicy mexican sala...,25,30-minutes-or-less time-to-make course main-in...,0.803729
7956,390966,sauteed green beans with almonds,15,15-minutes-or-less time-to-make course main-in...,0.802548


## 17. Interpretation and justification (short)

I evaluated models on 1000 users with a leave-one-out setup (predict the user’s last liked recipe from earlier likes). This is strict, so low hit rates are expected.

- **Text models:** TF-IDF and BERT (when available) consistently outperform BoW, which supports the idea that better text representation matters.
- **Hybrid weights:** I used a grid search over multiple weight combinations instead of one fixed setting. The best weights are selected by `HitRate@10` and then re-evaluated on the full user set.
- **Baselines:** Popularity remains very strong on this protocol, which is normal in real interaction data where head items dominate.
- **CF baseline:** SVD is now evaluated with the same metric function (`HitRate@10`, `Recall@10`, `MRR@10`) for fair comparison.
- **Diversity tradeoff:** Hybrid generally improves diversity compared with pure text, even when hit-rate gains are modest.

Main takeaway: stronger text features help, hybrid adds useful variety, and popularity is a tough benchmark under next-item metrics.

## 18. Conclusion

This notebook now addresses the main gaps directly:

1. **BERT validity:** BERT embeddings are generated in a subprocess to avoid kernel DLL failures. If BERT cannot run, results are explicitly labeled as fallback (not claimed as true BERT).
2. **Automated chatbot:** I added an automated chatbot flow (LLM-assisted when API is available, rule-based fallback otherwise) that converts user text into a retrieval query.
3. **Weighting strategy:** Hybrid weights are tuned with a small grid search, and the best setting is selected by metrics instead of fixed manually.
4. **Baseline fairness:** Random, Popular, and CF are evaluated with the same metric function and the same protocol.

Overall, the pipeline is now more credible: comparisons are transparent, baselines are consistent, and the chatbot section is both manual and automated.